In [1]:
# MongoDB Atlas Connection

import os
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError("Project root not found")

load_dotenv(PROJECT_ROOT / ".env", override=True)

mongo_uri = os.getenv("MONGODB_URI")

if not mongo_uri:
    raise ValueError("MONGODB_URI is missing")

client = MongoClient(
    mongo_uri,
    serverSelectionTimeoutMS=10000
)

client.admin.command("ping")

db = client["olist_db"]

print("MongoDB Atlas connected successfully!")
print("Database:", db.name)

MongoDB Atlas connected successfully!
Database: olist_db


In [8]:
db = client["olist_db"]

collections = db.list_collection_names()

print("Danh sách collection:")
for col in collections:
    print("-", col)

Danh sách collection:
- geolocation
- orders
- products
- sellers
- customers


In [9]:
print("Số documents trong từng collection:\n")

for col in db.list_collection_names():
    count = db[col].count_documents({})
    print(f"{col:<20} : {count:,}")

Số documents trong từng collection:

geolocation          : 19,015
orders               : 99,441
products             : 32,951
sellers              : 3,095
customers            : 99,441


In [10]:
for col in db.list_collection_names():
    print(f"\n===== {col.upper()} =====")
    
    sample = db[col].find_one()
    
    if sample:
        print("Các field:")
        for field in sample.keys():
            print(" -", field)


===== GEOLOCATION =====
Các field:
 - _id
 - geolocation_zip_code_prefix
 - geolocation_lat
 - geolocation_lng
 - geolocation_city
 - geolocation_state

===== ORDERS =====
Các field:
 - _id
 - order_id
 - customer_id
 - order_status
 - order_purchase_timestamp
 - order_approved_at
 - order_delivered_carrier_date
 - order_delivered_customer_date
 - order_estimated_delivery_date
 - items
 - payments
 - reviews

===== PRODUCTS =====
Các field:
 - _id
 - product_id
 - product_category_name
 - product_name_length
 - product_description_length
 - product_photos_qty
 - product_weight_g
 - product_length_cm
 - product_height_cm
 - product_width_cm
 - product_category_name_english

===== SELLERS =====
Các field:
 - _id
 - seller_id
 - seller_zip_code_prefix
 - seller_city
 - seller_state

===== CUSTOMERS =====
Các field:
 - _id
 - customer_id
 - customer_unique_id
 - customer_zip_code_prefix
 - customer_city
 - customer_state


In [11]:
sample_order = db["orders"].find_one()

print("===== ORDERS =====")

for key, value in sample_order.items():
    print(f"{key:<35} -> {type(value).__name__}")

===== ORDERS =====
_id                                 -> ObjectId
order_id                            -> str
customer_id                         -> str
order_status                        -> str
order_purchase_timestamp            -> str
order_approved_at                   -> str
order_delivered_carrier_date        -> str
order_delivered_customer_date       -> str
order_estimated_delivery_date       -> str
items                               -> list
payments                            -> list
reviews                             -> list


In [12]:
for key, value in sample_order.items():
    if isinstance(value, (list, dict)):
        print(f"\n===== {key} =====")
        print(value)


===== items =====
[{'order_item_id': 1, 'product_id': '595fac2a385ac33a80bd5114aec74eb8', 'seller_id': '289cdb325fb7e7f891c38608bf9e0962', 'shipping_limit_date': '2018-07-30T03:24:27', 'price': 118.7, 'freight_value': 22.76}]

===== payments =====
[{'payment_sequential': 1, 'payment_type': 'boleto', 'payment_installments': 1, 'payment_value': 141.46}]

===== reviews =====
[{'review_id': '8d5266042046a06655c8db133d120ba5', 'review_score': 4, 'review_comment_title': 'Muito boa a loja', 'review_comment_message': 'Muito bom o produto.', 'review_creation_date': '2018-08-08T00:00:00', 'review_answer_timestamp': '2018-08-08T18:37:50'}]


In [13]:
import pandas as pd

df_customers = pd.DataFrame(list(db["customers"].find()))
df_products = pd.DataFrame(list(db["products"].find()))
df_sellers = pd.DataFrame(list(db["sellers"].find()))
df_geolocation = pd.DataFrame(list(db["geolocation"].find()))

print("customers   :", df_customers.shape)
print("products    :", df_products.shape)
print("sellers     :", df_sellers.shape)
print("geolocation :", df_geolocation.shape)

customers   : (99441, 6)
products    : (32951, 11)
sellers     : (3095, 5)
geolocation : (19015, 6)


In [14]:
orders_raw = list(db["orders"].find())

df_orders = pd.DataFrame(orders_raw)

print("orders:", df_orders.shape)
print(df_orders.columns.tolist())

orders: (99441, 12)
['_id', 'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'items', 'payments', 'reviews']


In [15]:
# Tách order_items
items_data = []

for order in orders_raw:
    order_id = order["order_id"]
    
    for item in order.get("items", []):
        row = item.copy()
        row["order_id"] = order_id
        items_data.append(row)

df_order_items = pd.DataFrame(items_data)

print("order_items:", df_order_items.shape)
df_order_items.head()

order_items: (112650, 7)


,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_id
0,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30T03:24:27,118.70,22.76,53cdb2fc8bc7dce0b6741e2150273451
1,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23T19:45:59,45.00,27.20,949d5b44dbf5de918fe9c16f97b45f8a
2,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19T20:31:37,19.90,8.72,ad21c59c0840e6cb83a9ceb5573f8159
3,1,4520766ec412348b8d4caa5e8a18c464,16090f2ca825584b5a147ab24aa30c86,2017-05-22T13:22:11,59.99,15.17,6514b8ad8028c9f2cc2374ded245783f
4,1,08574b074924071f4e201e151b152b4e,001cca7ae9ae17fb1caed9dfb1094831,2017-05-22T19:50:18,99.00,30.53,e6ce16cb79ec1d90b1da9085a6118aeb


In [16]:
# Tách order_payments
payments_data = []

for order in orders_raw:
    order_id = order["order_id"]
    
    for payment in order.get("payments", []):
        row = payment.copy()
        row["order_id"] = order_id
        payments_data.append(row)

df_order_payments = pd.DataFrame(payments_data)

print("order_payments:", df_order_payments.shape)
df_order_payments.head()

order_payments: (103886, 5)


,payment_sequential,payment_type,payment_installments,payment_value,order_id
0,1,boleto,1,141.46,53cdb2fc8bc7dce0b6741e2150273451
1,1,credit_card,1,72.20,949d5b44dbf5de918fe9c16f97b45f8a
2,1,credit_card,1,28.62,ad21c59c0840e6cb83a9ceb5573f8159
3,1,credit_card,3,75.16,6514b8ad8028c9f2cc2374ded245783f
4,1,credit_card,1,259.06,e6ce16cb79ec1d90b1da9085a6118aeb


In [17]:
# Tách order_reviews
reviews_data = []

for order in orders_raw:
    order_id = order["order_id"]
    
    for review in order.get("reviews", []):
        row = review.copy()
        row["order_id"] = order_id
        reviews_data.append(row)

df_order_reviews = pd.DataFrame(reviews_data)

print("order_reviews:", df_order_reviews.shape)
df_order_reviews.head()

order_reviews: (99224, 7)


,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_id
0,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08T00:00:00,2018-08-08T18:37:50,53cdb2fc8bc7dce0b6741e2150273451
1,359d03e676b3c069f62cadba8dd3f6e8,5,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03T00:00:00,2017-12-05T19:21:58,949d5b44dbf5de918fe9c16f97b45f8a
2,e50934924e227544ba8246aeb3770dd4,5,NaN,NaN,2018-02-17T00:00:00,2018-02-18T13:02:51,ad21c59c0840e6cb83a9ceb5573f8159
3,07d67dd06ed5f88bef11ef6b464e79ae,5,NaN,NaN,2017-05-27T00:00:00,2017-05-28T02:59:57,6514b8ad8028c9f2cc2374ded245783f
4,15898b543726a832d4137fbef5d1d00e,1,NaN,Aguardando retorno da loja,2017-05-30T00:00:00,2017-05-30T23:13:47,e6ce16cb79ec1d90b1da9085a6118aeb


In [18]:
df_orders_sql = df_orders.drop(
    columns=["_id", "items", "payments", "reviews"]
)

print("orders SQL:", df_orders_sql.shape)
df_orders_sql.head()

orders SQL: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37,2018-07-26T03:24:27,2018-07-26T14:31:00,2018-08-07T15:27:45,2018-08-13T00:00:00
1,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06,2017-11-18T19:45:59,2017-11-22T13:39:59,2017-12-02T00:28:42,2017-12-15T00:00:00
2,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39,2018-02-13T22:20:29,2018-02-14T19:46:34,2018-02-16T18:17:02,2018-02-26T00:00:00
3,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30,2017-05-16T13:22:11,2017-05-22T10:07:46,2017-05-26T12:55:51,2017-06-07T00:00:00
4,e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,delivered,2017-05-16T19:41:10,2017-05-16T19:50:18,2017-05-18T11:40:40,2017-05-29T11:18:31,2017-06-07T00:00:00


In [19]:
# SQL Server Connection

import os
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

server = os.getenv("SQL_SERVER")
database = os.getenv("SQL_DATABASE")
driver = os.getenv("SQL_DRIVER")

if not all([server, database, driver]):
    raise ValueError("Missing SQL Server configuration in .env")

conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# Test SQL Server connection
with pyodbc.connect(conn_str, timeout=10) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT DB_NAME()")
    print("Connected to:", cursor.fetchone()[0])

# Create SQLAlchemy engine
connection_url = URL.create(
    "mssql+pyodbc",
    query={"odbc_connect": conn_str}
)

engine = create_engine(
    connection_url,
    fast_executemany=True
)

print("SQLAlchemy Engine ready!")

Connected to: Capstone_DA_ADV_04
SQLAlchemy Engine ready!


In [20]:
# Check SQL Server tables

from sqlalchemy import inspect

inspector = inspect(engine)
table_names = inspector.get_table_names()

print("SQL Server Tables:")
for table in table_names:
    print("-", table)

print(f"\nTotal tables: {len(table_names)}")

SQL Server Tables:
- customers
- geolocation
- order_items
- order_payments
- order_reviews
- orders
- products
- sellers

Total tables: 8


In [21]:
# Validate SQL Server table records

from sqlalchemy import text

print("SQL Server Data Validation\n")

with engine.connect() as conn:
    for table_name in table_names:
        result = conn.execute(
            text(f"SELECT COUNT(*) FROM [{table_name}]")
        )
        count = result.scalar()
        print(f"{table_name}: {count:,} records")

SQL Server Data Validation

customers: 99,441 records
geolocation: 19,015 records
order_items: 112,650 records
order_payments: 103,886 records
order_reviews: 99,224 records
orders: 99,441 records
products: 32,951 records
sellers: 3,095 records


In [22]:
# Bỏ _id của MongoDB nếu còn tồn tại
for df in [df_customers, df_products, df_sellers, df_geolocation]:
    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

# Chuyển các cột datetime của orders
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    df_orders_sql[col] = pd.to_datetime(df_orders_sql[col], errors="coerce")

# Datetime của order_items
df_order_items["shipping_limit_date"] = pd.to_datetime(
    df_order_items["shipping_limit_date"],
    errors="coerce"
)

# Datetime của reviews
review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_cols:
    df_order_reviews[col] = pd.to_datetime(
        df_order_reviews[col],
        errors="coerce"
    )

print("Chuẩn hóa datatype hoàn tất!")

Chuẩn hóa datatype hoàn tất!


C:\Users\t14\AppData\Local\Temp\ipykernel_8720\857248057.py:4: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["_id"], inplace=True)


In [23]:
print("ORDERS")
print(df_orders_sql.dtypes)

print("\nORDER ITEMS")
print(df_order_items.dtypes)

print("\nORDER REVIEWS")
print(df_order_reviews.dtypes)

ORDERS
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

ORDER ITEMS
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
order_id                          str
dtype: object

ORDER REVIEWS
review_id                             str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
order_id                           

In [25]:
# Load DataFrames into SQL Server

from sqlalchemy import inspect

tables = {
    "customers": df_customers,
    "geolocation": df_geolocation,
    "orders": df_orders,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "products": df_products,
    "sellers": df_sellers
}

for table_name, df in tables.items():
    inspector = inspect(engine)

    if inspector.has_table(table_name):
        print(f"{table_name}: Table already exists. Skipped.")
        continue

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="fail",
        index=False,
        chunksize=1000
    )

    print(f"{table_name}: Loaded {len(df):,} records")

customers: Table already exists. Skipped.
geolocation: Table already exists. Skipped.
orders: Table already exists. Skipped.
order_items: Table already exists. Skipped.
order_payments: Table already exists. Skipped.
order_reviews: Table already exists. Skipped.
products: Table already exists. Skipped.
sellers: Table already exists. Skipped.


In [26]:
# Final SQL Server Data Validation

from sqlalchemy import text

for table_name, dataframe in tables.items():
    with engine.connect() as conn:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM [{table_name}]")
        ).scalar()

    expected = len(dataframe)
    status = "OK" if count == expected else "MISMATCH"

    print(f"{table_name}: {count:,}/{expected:,} - {status}")

customers: 99,441/99,441 - OK
geolocation: 19,015/19,015 - OK
orders: 99,441/99,441 - OK
order_items: 112,650/112,650 - OK
order_payments: 103,886/103,886 - OK
order_reviews: 99,224/99,224 - OK
products: 32,951/32,951 - OK
sellers: 3,095/3,095 - OK
